# 附录B 数学与概率回顾 —— 配套代码

对应正文 `book/appendix/b-math-review.md`。用 numpy/scipy 演示关键数学概念，离线可跑。

> 运行前请先生成内置数据：`uv run python scripts/make_sample_data.py`。

In [ ]:
# === 在线运行引导（Google Colab）。本地 / Binder 会自动跳过 ===
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess

    if not os.path.isdir("src/fds"):  # 尚未进入仓库
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/albertandking/financial-data-science", "fds_book"],
                       check=True)
        os.chdir("fds_book")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
        print("已就绪：仓库已克隆、fds 已安装、内置数据随仓库一并克隆。")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from fds import load_sample_prices, daily_returns, set_chinese_font

set_chinese_font()
rng = np.random.default_rng(0)

## B.1 线性代数：组合收益、组合方差（二次型）

In [ ]:
rets = daily_returns(load_sample_prices())
mu = rets.mean().values * 252          # 年化期望收益向量
Sigma = rets.cov().values * 252        # 年化协方差矩阵
w = np.array([0.25, 0.25, 0.25, 0.25]) # 等权

port_mu = w @ mu                       # 组合收益 = w^T mu
port_var = w @ Sigma @ w               # 组合方差 = w^T Sigma w（二次型）
print(f'组合年化收益 w^T mu = {port_mu:.4f}')
print(f'组合年化波动 sqrt(w^T Sigma w) = {np.sqrt(port_var):.4f}')

## B.1.6 特征值分解与 PCA：第一主成分≈市场因子

对相关矩阵做谱分解，最大特征值方向解释的方差占比。

In [ ]:
corr = rets.corr().values
eigvals, eigvecs = np.linalg.eigh(corr)   # 对称矩阵谱分解
order = np.argsort(eigvals)[::-1]
eigvals = eigvals[order]
print('特征值（降序）:', np.round(eigvals, 3))
print(f'第一主成分解释方差占比: {eigvals[0]/eigvals.sum():.1%}')
print('第一主成分载荷（各股票同号→市场因子）:', np.round(eigvecs[:, order[0]], 3))

## B.2 最优化：梯度下降求最小值

以 $f(x)=(x-3)^2+2$ 为例，负梯度方向迭代收敛到 $x^*=3$。

In [ ]:
f = lambda x: (x - 3) ** 2 + 2
grad = lambda x: 2 * (x - 3)

x, eta, path = -2.0, 0.1, []
for _ in range(50):
    path.append(x)
    x = x - eta * grad(x)        # 梯度下降更新
print(f'收敛到 x = {x:.4f}（真实最优 3）')

xs = np.linspace(-3, 9, 200)
plt.plot(xs, f(xs), label='f(x)')
plt.scatter(path, [f(p) for p in path], c='crimson', s=15, label='迭代轨迹')
plt.title('梯度下降'); plt.legend(); plt.tight_layout(); plt.show()

## B.2.4 拉格朗日：全局最小方差组合（GMV）解析解

$\mathbf{w}_{GMV} = \dfrac{\Sigma^{-1}\mathbf{1}}{\mathbf{1}^\top\Sigma^{-1}\mathbf{1}}$

In [ ]:
ones = np.ones(len(mu))
inv = np.linalg.inv(Sigma)
w_gmv = inv @ ones / (ones @ inv @ ones)
print('GMV 权重:', dict(zip(rets.columns, np.round(w_gmv, 3))))
print(f'GMV 年化波动 = {np.sqrt(w_gmv @ Sigma @ w_gmv):.4f}（应小于等权 {np.sqrt(port_var):.4f}）')

## B.3 概率：矩、偏度、峰度（厚尾的证据）

In [ ]:
x = rets['TECH']
print(f'均值={x.mean():.5f}  标准差={x.std():.5f}')
print(f'偏度={stats.skew(x):.3f}  超额峰度={stats.kurtosis(x):.3f}（正态为0；>0即厚尾）')

## B.3.6 中心极限定理：均值的分布趋于正态

In [ ]:
# 从一个非正态（指数）总体反复抽样求均值，观察均值分布形态
pop = rng.exponential(1.0, size=200_000)
means = [rng.choice(pop, size=50).mean() for _ in range(2000)]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(pop, bins=60); ax[0].set_title('总体：指数分布（右偏）')
ax[1].hist(means, bins=40, density=True, alpha=0.7)
ax[1].set_title('样本均值分布≈正态（CLT）')
plt.tight_layout(); plt.show()

## B.4 统计推断：t 检验与 p 值

检验 TECH 的日均收益是否显著异于 0（$H_0: \mu=0$）。

In [ ]:
t_stat, p_val = stats.ttest_1samp(x, popmean=0.0)
print(f't 统计量={t_stat:.3f}  p 值={p_val:.3f}')
print('结论:', '拒绝 H0（日均收益显著≠0）' if p_val < 0.05 else '不能拒绝 H0（与0无显著差异）')
print('→ 低信噪比下，日均收益通常难以显著（见正文第1章）')

## B.5 回归：OLS 闭式解 $(X^\top X)^{-1}X^\top y$ 验证

In [ ]:
# 用 BANK 收益回归 TECH 收益（截距+斜率），手算 OLS 并与 numpy 对照
y = rets['TECH'].values
X = np.column_stack([np.ones(len(y)), rets['BANK'].values])
beta = np.linalg.inv(X.T @ X) @ X.T @ y
print(f'OLS 闭式解: 截距={beta[0]:.5f}, 斜率={beta[1]:.4f}')
beta_np = np.linalg.lstsq(X, y, rcond=None)[0]
print('与 np.linalg.lstsq 一致:', np.allclose(beta, beta_np))

## 小结

本附录的数学贯穿全书：二次型→组合方差（第14章）、谱分解→因子（第7章）、
梯度下降→模型训练（第9/12章）、矩与厚尾→风险（第4/5章）、假设检验→因子显著性
与回测陷阱（第6/7/8/15章）、OLS→回归与监督学习（第7/8/9章）。